# Understanding the data and experimenting with collection of external data

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

## Import and view data

In [4]:
cross_border_payments = pd.read_csv("../data/cross_border_payments.csv")
trade_finance = pd.read_csv("../data/trade_finance.csv")
transactional_banking = pd.read_csv("../data/transactional_banking.csv")

In [7]:
cross_border_payments.head()

,transaction_id,entity_id,entity_name,sector,date,direction,currency_pair,value_zar,counterparty_country,corridor_type,beneficiary_name,reference,memo
0,XBP63220455,E01,BHP Group,mining,2023-07-01,inbound,USD/ZAR,2541553.55,Switzerland,intercompany,BHP Group Switzerland Ltd,INTERCO-730855,NaN
1,XBP14207725,E11,Pepkor Holdings,consumer,2023-07-01,outbound,USD/ZAR,407839.87,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-827118,NaN
2,XBP66460952,E11,Pepkor Holdings,consumer,2023-07-01,outbound,CNY/ZAR,72148.47,Angola,intercompany,Pepkor Holdings Angola Ltd,INTERCO-488519,NaN
3,XBP45973312,E11,Pepkor Holdings,consumer,2023-07-01,outbound,GBP/ZAR,53285.65,Namibia,intercompany,Pepkor Holdings Namibia Ltd,INTERCO-585129,NaN
4,XBP13173829,E11,Pepkor Holdings,consumer,2023-07-01,inbound,AED/ZAR,2858193.91,Japan,trade,Continental Resources Trading,TRADE-568825,NaN


In [8]:
trade_finance.head()

,instrument_id,entity_id,entity_name,sector,date,instrument_type,direction,tenor_days,value_zar,counterparty_country,commodity_or_contract_type,status,beneficiary_name,reference,memo
0,TF67401938,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,60,13499920.55,United Arab Emirates,agri_produce,issued,Silverline Trading Co.,LC-471415,NaN
1,TF91455580,E01,BHP Group,mining,2023-07-01,letters_of_credit,export,365,558304.97,Switzerland,iron_ore,settled,Global Commodities Marketing,LC-266865,NaN
2,TF31370953,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,30,4042335.97,United Kingdom,agri_produce,settled,Pacific International Trading House,LC-946978,NaN
3,TF13634137,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,import,365,171042.89,China,platinum_group_metals,active,Silverline Resources Trading,LC-553503,NaN
4,TF86438695,E10,Bid Corporation,consumer,2023-07-01,letters_of_credit,export,120,331531.78,Netherlands,electronics,settled,Silverline Resources Trading,LC-260628,NaN


In [9]:
transactional_banking.head()

,transaction_id,entity_id,entity_name,sector,date,leg_type,direction,amount_zar,currency,channel,beneficiary_name,reference,memo
0,TXN40610803,E01,BHP Group,mining,2023-07-01,collections,inbound,63878.473869,ZAR,EFT,Continental Metals Trading House,INV-662227,NaN
1,TXN12547643,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,53220.970000,ZAR,EFT,Sunrise Cold Chain Logistics,INV-591371,NaN
2,TXN37710224,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,12309.970000,ZAR,SWIFT,Sunrise Cold Chain Logistics,PO-687736,NaN
3,TXN65618575,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,6308.300000,ZAR,Internal Transfer,Sunrise Cold Chain Logistics,INV-139304,NaN
4,TXN50222056,E11,Pepkor Holdings,consumer,2023-07-01,supplier_payments,outbound,55346.750000,ZAR,Internal Transfer,Cape Wholesale Distributors,INV-556985,NaN


In [32]:
list(set(transactional_banking.entity_name.unique()) | \
    set(cross_border_payments.entity_name.unique()) | \
    set(trade_finance.entity_name.unique()))

['Sanlam',
 'Vodacom Group',
 'OUTsurance Group',
 'Valterra Platinum',
 'Naspers',
 'Shaftesbury Capital plc',
 'NEPI Rockcastle',
 'Gold Fields',
 'BHP Group',
 'Glencore',
 'AngloGold Ashanti',
 'Shoprite Holdings',
 'Clicks Group',
 'Prosus',
 'Bid Corporation',
 'Pepkor Holdings',
 'Anglo American',
 'Aspen Pharmacare',
 'The Bidvest Group',
 'MTN Group']

Interesting, we only have 20 companies...

#TODO: Double check
My understanding of the data is as follows:
- Transactional: Actual payments made by companies
- Cross border payments: Message sent between banks, used when money is transfered internationally 
(e.g. Syn bank sends a swift message to Barclays that Pepkor transfered 1000 pounds to jane street)
- Trade Finance: Basically communication to ensure that buyers and sellers are protected. It'll 
store things like company a is buying product from company b, then a credit note is reached out to 
tell company b that it will get payed once the product is shipped.

**This means that the same transaction can appear in all 3 datasets, so we need to figure out how 
to remove the same transaction**

**We need to find out what the core product pillars are for wich we are predicting wallet share**

## Retrieving external data

We need to get external data to actually predict wallet size

The goal of this section is not to retrieve data, but more to experiment on possible ways to get 
the data. Once methods are finalized, a Python script will be written to retrieve external data.

In [4]:
# Tickers string for companies, used with yahoo finance
tickers_str = "SLM.JO VOD.JO OUT.JO VAL.JO NPN.JO SHC.JO NRP.JO GFI.JO BHG.JO GLN.JO ANG.JO SHP.JO " \
"CLS.JO PRX.JO BVT.JO PPH.JO AGL.JO APN.JO BTI.JO MTN.JO"

#Used for getting ticker data
company_tickers = {
    "Sanlam": "SLM.JO",
    "Vodacom Group": "VOD.JO",
    "OUTsurance Group": "OUT.JO",
    "Valterra Platinum": "VAL.JO",
    "Naspers": "NPN.JO",
    "Shaftesbury Capital plc": "SHC.JO",
    "NEPI Rockcastle": "NRP.JO",
    "Gold Fields": "GFI.JO",
    "BHP Group": "BHG.JO",
    "Glencore": "GLN.JO",
    "AngloGold Ashanti": "ANG.JO",
    "Shoprite Holdings": "SHP.JO",
    "Clicks Group": "CLS.JO",
    "Prosus": "PRX.JO",
    "Bid Corporation": "BVT.JO",
    "Pepkor Holdings": "PPH.JO",
    "Anglo American": "AGL.JO",
    "Aspen Pharmacare": "APN.JO",
    "The Bidvest Group": "BTI.JO",
    "MTN Group": "MTN.JO",
}

In [2]:
import yfinance as yf

portfolio = yf.Tickers(tickers_str)

In [9]:
dir(portfolio.tickers[company_tickers["Sanlam"]])

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_analysis',
 '_data',
 '_download_options',
 '_earnings',
 '_earnings_dates',
 '_expirations',
 '_fast_info',
 '_fetch_ticker_tz',
 '_financials',
 '_fundamentals',
 '_funds_data',
 '_get_earnings_dates_using_scrape',
 '_get_earnings_dates_using_screener',
 '_get_ticker_tz',
 '_holders',
 '_isin',
 '_lazy_load_price_history',
 '_message_handler',
 '_news',
 '_options2df',
 '_price_history',
 '_quote',
 '_shares',
 '_tz',
 '_underlying',
 'actions',
 'analyst_price_targets',
 'balance_sheet',
 'balancesheet',
 'calendar',
 'capital_gains',
 'cash_flow',
 'cashflow',
 'dividends',
 'earnings',
 'earnings_d

In [2]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.scripts.data_processing import Data_Processor

In [3]:
my_data_processor = Data_Processor()

In [4]:
company_lvl_df, sens_df = my_data_processor.extract_external_data_from_pdfs('../data/downloads')

12:29:12 | INFO | Pdf paths: [PosixPath('../data/downloads/AngloGold_Ashanti/annual_report__2025__annual_report.pdf')]
12:30:07 | INFO | Pdf paths: [PosixPath('../data/downloads/AngloGold_Ashanti/financial_statements__2025__financial_statements.pdf')]
12:30:57 | INFO | Pdf paths: [PosixPath('../data/downloads/AngloGold_Ashanti/interim_results__2026__interim_results.pdf')]
12:31:28 | INFO | Pdf paths: [PosixPath('../data/downloads/AngloGold_Ashanti/results_presentation__2026__results_presentation.pdf')]
12:31:50 | INFO | Pdf paths: [PosixPath('/tmp/merged_sens_pqdgbd2m/AngloGold_Ashanti__SENS_merged.pdf')]
12:32:55 | INFO | Pdf paths: [PosixPath('../data/downloads/Anglo_American/financial_statements__2025__aa-annual-report-full-2025.pdf')]
12:34:00 | INFO | Pdf paths: [PosixPath('../data/downloads/Anglo_American/interim_results__2026__half-year-results-2026-factsheet.pdf')]
Annotation sizes differ: 2 vs. 0
Annotation sizes differ: 2 vs. 0
Annotation sizes differ: 2 vs. 0
12:34:18 | INFO

In [8]:
company_lvl_df.columns

Index(['company', 'report_date', 'reporting_currency', 'reporting_unit',
       'source_document', 'revenue', 'cost_of_sales', 'inventory',
       'trade_receivables', 'trade_payables', 'cash_and_cash_equivalents',
       'operating_cash_flow', 'total_debt', 'short_term_debt',
       'long_term_debt', 'debt_maturity_schedule', 'debt_due_within_12_months',
       'debt_due_12_to_24_months', 'undrawn_committed_facilities',
       'finance_costs', 'capital_expenditure', 'currencies_exposed_to',
       'employee_expenses', 'employee_count', 'dividends_paid', 'tax_paid',
       'countries_of_operation', 'foreign_subsidiaries',
       'major_customers_suppliers', 'commodity_exposure',
       'interest_rate_exposure', 'bond_issues', 'bond_maturity_dates',
       'credit_facilities_and_lenders', 'major_contractual_commitments',
       'order_book_project_pipeline', 'share_price', 'share_return',
       'ownership_major_shareholders', 'extra_notes', 'foreign_revenue',
       'revenue_by_geograp

In [6]:
sens_df

,company,announcement_date,title,source_document,source_url,event_type,event_value,currency,counterparty,target_or_asset,country,expected_completion_date,banking_opportunities,opportunity_summary,extra_notes
0,AngloGold Ashanti plc,30 April 2025,AngloGold Ashanti Agrees Sale of Côte d’Ivoire...,AngloGold Ashanti Agrees Sale of Côte d’Ivoire...,NaN,disposal,1.750000e+08,USD,Resolute Mining Limited,Doropo Project and ABC Project,Côte d’Ivoire,1 May 2025,"[fx, payments, collections, corporate_finance,...",Sale of Doropo and ABC projects for US$175 mil...,"In connection with the sale, AGA will acquire ..."
1,AngloGold Ashanti plc,2 June 2025,AngloGold Ashanti Agrees the Sale of the Miner...,AngloGold Ashanti Agrees the Sale of the Miner...,NaN,disposal,7.600000e+07,USD,Aura Minerals Inc.,Mineração Serra Grande mine,Brazil,Q3 2025,"[fx, payments, collections, corporate_finance]",Disposal of Mineração Serra Grande mine in Bra...,Includes deferred consideration payments equiv...
2,AngloGold Ashanti plc,16 July 2025,AngloGold Ashanti Agrees to Acquire Augusta Go...,AngloGold Ashanti Agrees to Acquire Augusta Go...,NaN,acquisition,1.520000e+08,CAD,Augusta Gold Corp.,Augusta Gold Corp.,United States,Fourth quarter of 2025,"[credit, corporate_finance, fx, payments, debt...",Acquisition of Augusta Gold Corp. for approxim...,AngloGold Ashanti will also provide funds for ...
3,AngloGold Ashanti plc,8 May 2026,AngloGold Ashanti Q1 31 March 2026 Earnings Re...,AngloGold Ashanti Q1 31 March 2026 Earnings Re...,NaN,share_buyback,2.000000e+09,USD,NaN,Ordinary shares,United States,NaN,"[corporate_finance, investor_services, payments]",Proposed share repurchase programme of up to U...,Approved by the Board on 7 May 2026 and approv...
4,Anglo American plc,2025-01-29,Anglo American completes sale of minority inte...,Anglo American completes sale of minority inte...,NaN,disposal,1.600000e+09,AUD,Zashvin Pty Ltd,33.3% minority interest in Jellinbah Group Pty...,Australia,NaN,"[payments, collections, fx, liquidity_management]",Cash proceeds from the disposal create immedia...,Anglo American completed the sale of its 33.3%...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,Vodacom Group Limited,4 December 2025,ACQUISITION OF A FURTHER 20% INTEREST IN SAFAR...,ACQUISITION OF A FURTHER 20% INTEREST IN SAFAR...,NaN,acquisition,2.100000e+09,USD,Government of Kenya and Vodafone International...,Safaricom PLC,Kenya,Q1 2026,"[credit, debt_capital_markets, corporate_finan...",Major cross-border acquisition of a 20% stake ...,Acquisition of an effective 20% interest in Sa...
88,Vodacom Group Limited,19 May 2025,Vodacom Group reviewed annual results and cash...,Vodacom Group reviewed annual results and cash...,https://senspdf.jse.co.za/documents/2025/jse/i...,dividend,NaN,ZAR,NaN,NaN,South Africa,23 June 2025,"[payments, investor_services]",Large-scale dividend payout requiring corporat...,Full year dividend of 620 cps declared (final ...
89,Vodacom Group Limited,7 November 2025,Vodacom Group reviewed interim results and cas...,Vodacom Group reviewed interim results and cas...,https://senspdf.jse.co.za/documents/2025/jse/i...,dividend,NaN,ZAR,NaN,NaN,South Africa,1 December 2025,"[payments, investor_services]",Interim dividend distribution creating cash ma...,Gross interim dividend number 33 of 330 cents ...
90,Vodacom Group Limited,8 May 2026,Vodacom Group reviewed annual results and cash...,Vodacom Group reviewed annual results and cash...,https://senspdf.jse.co.za/documents/2026/jse/i...,dividend,NaN,ZAR,NaN,NaN,South Africa,22 June 2026,"[payments, investor_services]",Full year dividend distribution offering trans...,Final dividend number 34 of 405 cents per ordi...


In [ ]:
my

,company,report_date,reporting_currency,reporting_unit,source_document,revenue,cost_of_sales,inventory,trade_receivables,trade_payables,cash_and_cash_equivalents,operating_cash_flow,total_debt,short_term_debt,long_term_debt,debt_maturity_schedule,debt_due_within_12_months,debt_due_12_to_24_months,undrawn_committed_facilities,finance_costs,capital_expenditure,currencies_exposed_to,employee_expenses,employee_count,dividends_paid,tax_paid,countries_of_operation,foreign_subsidiaries,major_customers_suppliers,commodity_exposure,interest_rate_exposure,bond_issues,bond_maturity_dates,credit_facilities_and_lenders,major_contractual_commitments,order_book_project_pipeline,share_price,share_return,ownership_major_shareholders,extra_notes,foreign_revenue,revenue_by_geography,security_collateral_on_debt,credit_rating,fx_hedging_policy,contingent_liabilities,exports_exposure,commodity_derivatives,interest_rate_derivatives,fx_derivative_notional,guarantees_outstanding,receivable_days,payable_days,inventory_days,market_capitalisation,foreign_currency_liabilities,assets_under_management,imports_exposure,foreign_currency_assets
0,AngloGold Ashanti plc,30 June 2026,USD,millions,Q2 2026 Earnings Results,3.104000e+03,1528.0,1285.0,557.0,1048.0,2.769000e+03,1.432000e+03,1.778000e+03,12.0,1559.0,"Within one year: $19 million; Between two and five years: $1,732 million; After five years: $293 million.",12.0,0.0,1465.0,38.0,5.490000e+02,"[USD, ZAR, AUD, BRL, ARS, EGP]",8.800000e+02,41416.0,8.270000e+02,542.0,"[United States of America, Colombia, Brazil, Argentina, Guinea, Ghana, Tanzania, Egypt, DRC, Australia]","[AngloGold Ashanti Australia Limited, AngloGold Ashanti Holdings plc, AngloGold Ashanti USA Incorporated, AngloGold Ashanti Córrego do Sítio Mineração S.A., AngloGold Ashanti (Ghana) Limited, AngloGold Ashanti (Iduapriem) Limited, Cerro Vanguardia S.A., Geita Gold Mining Limited, Société AngloGold Ashanti de Guinée S.A., Sukari Gold Mines Company]","[ANZ Investment Bank Ltd, Standard Chartered Bank, JP Morgan Chase NA New York, MKS Finance SA]",[Gold],Fixed and variable rate borrowings denominated in USD and Tanzanian shillings.,Repurchased approximately $666m principal amount of its outstanding bonds on 16 April 2026.,"[2028-11-01, 2030-10-01, 2040-04-15]","$1.4bn multi-currency revolving credit facility (RCF) undrawn, $65m Siguiri RCF undrawn, $489m Geita RCF fully drawn.","Contracted capital expenditure of $378 million within one year and $811 million not contracted for. Other purchase obligations of $1,686 million.","Key advanced projects include Arthur Gold Project (Merlin/Silicon) and North Bullfrog in Nevada, USA, and Quebradona in Colombia.",85.28,2.750,"[Public Investment Corporation of South Africa, BlackRock, Inc.]","Q2 2026 earnings results presentation. Net cash balance of $991m achieved. Executed proactive liability management program, retiring $666m of long-dated bonds in April 2026. Proposed $2bn share repurchase program approved.",10768.0,"Africa: $7,152m; Australia: $1,876m; Americas: $1,740m.","Unsecured borrowings and bonds, with specific asset pledges mentioned only for minor security ($7m in tangible assets).",BB+ / Baa3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Anglo American plc,2025-12-31,USD,millions,Integrated Annual Report 2025,1.854600e+04,17194.0,3819.0,2390.0,2479.0,6.436000e+03,7.005000e+03,1.548100e+04,1075.0,14406.0,"Short term borrowings of $1,075 million and medium/long term borrowings of $14,406 million as of 31 December 2025.",1075.0,NaN,6000.0,862.0,3.322000e+03,"[USD, ZAR, AUD, CLP, BRL, EUR, GBP]",2.669000e+03,43000.0,3.440000e+02,1329.0,"[United Kingdom, Chile, South Africa, Australia, Peru, Brazil, Zimbabwe, Botswana, Namibia, Canada, Finland]","[Anglo American Sur S.A., Anglo American Quellaveco S.A., Kumba Iron Ore Limited, De Beers plc, Minas-Rio, Collahuasi]","[Stanwell Corporation, Enel X, Codelco, Vale, Mitsubishi Corporation]","[Copper, Iron ore, Manganese, Crop nutrients,

In [7]:
company_lvl_df.to_csv('company_lvl_df.csv')
sens_df.to_csv('sens_df.csv')

In [ ]:
company_res.model_dump_json()

str

In [32]:
#NOTE: Method now automatically dumps

company_json = json.loads(company_res.model_dump_json())
sens_json = json.loads(sens_res.model_dump_json())

In [35]:
pd.DataFrame(company_json["records"])

,company,report_date,reporting_currency,reporting_unit,source_document,revenue,cost_of_sales,inventory,trade_receivables,trade_payables,...,security_collateral_on_debt,major_contractual_commitments,order_book_project_pipeline,assets_under_management,market_capitalisation,share_price,share_return,enterprise_value,credit_rating,ownership_major_shareholders
0,Pepkor Holdings Limited,2025-09-30,ZAR,Millions,Pepkor Holdings Limited Annual Financial State...,95340.0,57388.0,18618.0,1551.0,12023.0,...,Long- and short-term investments at banking in...,Contracts for capital expenditure amounting to...,None,None,89800.0,2431.0,None,None,None,"[Public Investment Corporation (PIC), Titan Pr..."


In [37]:
pd.DataFrame(sens_json["events"])

,company,announcement_date,title,source_document,source_url,event_type,event_value,currency,counterparty,target_or_asset,country,expected_completion_date,banking_opportunities,opportunity_summary
0,Pepkor Holdings Limited,10 December 2025,LISTING OF NEW FINANCIAL INSTRUMENTS,LISTING OF NEW FINANCIAL INSTRUMENTS,None,bond_issue,1.750000e+09,ZAR,Nedbank Limited and Absa Bank Limited,PEP12 and PEP13 Senior Unsecured Floating Rate...,South Africa,NaN,"[debt_capital_markets, interest_rates]",Listing of PEP12 (ZAR 750 million) and PEP13 (...
1,Pepkor Holdings Limited,4 March 2026,LISTING OF NEW FINANCIAL INSTRUMENTS,LISTING OF NEW FINANCIAL INSTRUMENTS,None,bond_issue,1.130000e+09,ZAR,Nedbank Limited,PEP14 and PEP15 Senior Unsecured Floating Rate...,South Africa,NaN,"[debt_capital_markets, interest_rates]",Listing of PEP14 (ZAR 595 million) and PEP15 (...
2,Pepkor Holdings Limited,7 March 2025,LISTING OF NEW FINANCIAL INSTRUMENTS,LISTING OF NEW FINANCIAL INSTRUMENTS,None,bond_issue,2.083000e+09,ZAR,Rand Merchant Bank,PEP09 and PEP10 Senior Unsecured Floating Rate...,South Africa,NaN,"[debt_capital_markets, interest_rates, credit]",Pepkor raised R2.1 billion via an auction and ...
3,Pepkor Holdings Limited,28 March 2025,LISTING OF NEW FINANCIAL INSTRUMENT,LISTING OF NEW FINANCIAL INSTRUMENT,None,bond_issue,1.250000e+09,ZAR,Absa Bank Limited,PEP11 Senior Unsecured Floating Rate Notes,South Africa,NaN,"[debt_capital_markets, interest_rates, credit]",Placement of R1.25 billion of PEP11 notes to r...
4,Pepkor Holdings Limited,22 July 2026,PEPKOR TO ACQUIRE CONTROLLING STAKE IN TRANSFO...,PEPKOR TO ACQUIRE CONTROLLING STAKE IN TRANSFO...,None,acquisition,1.570000e+09,ZAR,Shop2Shop Proprietary Limited and S2S Africa H...,Shop2Shop Proprietary Limited and Flash Mobile...,South Africa,NaN,"[corporate_finance, credit, fx, payments]",Pepkor to acquire a controlling 57.1% stake in...
5,Pepkor Holdings Limited,25 March 2025,VOLUNTARY ANNOUNCEMENT RELATING TO THE ACQUISI...,VOLUNTARY ANNOUNCEMENT RELATING TO THE ACQUISI...,None,acquisition,NaN,ZAR,Retailability Proprietary Limited,"Legit, Swagga, Style and Boardmans businesses",South Africa,NaN,"[corporate_finance, credit, payments]","Acquisition of Legit, Swagga, Style and Boardm..."
6,Pepkor Holdings Limited,4 November 2025,VOLUNTARY ANNOUNCEMENT RELATING TO THE SUCCESS...,VOLUNTARY ANNOUNCEMENT RELATING TO THE SUCCESS...,None,acquisition,1.700000e+09,ZAR,Retailability Proprietary Limited,"Legit, Swagga, Style and Boardmans businesses",South Africa,2025-11-02,"[corporate_finance, credit, payments]",Successful implementation of the acquisition o...
